# TimesFM


In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ml_final_project


Mounted at /content/drive
/content/drive/MyDrive/ml_final_project


In [2]:
import os
if not os.path.exists('/content/ml_final_project'):
    !git clone -q https://github.com/ochiga07/ml_final_project.git /content/ml_final_project
import sys
sys.path.append('/content/ml_final_project')

from colab_setup import setup_project

drive_repo = setup_project(repo_url="https://github.com/ochiga07/ml_final_project.git")

import feature_pipeline
import importlib
importlib.reload(feature_pipeline)
from feature_pipeline import load_raw_data, run_feature_pipeline

from metrics import wmae


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install -q timesfm mlflow dagshub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/1

In [4]:
import os
import numpy as np
import pandas as pd
import timesfm
import mlflow
import dagshub
import warnings
warnings.filterwarnings('ignore')

if 'DAGSHUB_USER_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['DAGSHUB_USER_TOKEN'] = userdata.get('DAGSHUB_USER_TOKEN')
    except Exception:
        pass

dagshub.init(repo_owner='aochi23', repo_name='ml_final_project', mlflow=True)
mlflow.set_experiment('TimesFM_Training')


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=c05f19bb-7f99-43e8-a31d-0b5e3b8d10e1&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=814e820ea1bbd4736ac7a9f575a28f8d0f43011546547157e0fb558a69bea43d




Accessing as Saba0033

Initialized MLflow to track repo "aochi23/ml_final_project"

Repository aochi23/ml_final_project initialized!

<Experiment: artifact_location='mlflow-artifacts:/45373a9ac5a240f79db1a1f1d84ff202', creation_time=1783762237289, effective_trace_archival_retention=None, experiment_id='6', last_update_time=1783762237289, lifecycle_stage='active', name='TimesFM_Training', tags={}, trace_location=None, workspace='default'>

## Data


In [5]:
train, test, features, stores = load_raw_data(path=f'{drive_repo}/data/')
full_df = run_feature_pipeline(train, test, features, stores)

train_df = full_df[full_df['is_train'] == 1].drop(columns=['is_train'])
print(train_df.shape)


(421570, 39)


In [6]:
VAL_WEEKS = 12
MIN_HISTORY = 20

pairs = train_df.groupby(['Store', 'Dept'])
print(f'total pairs: {len(pairs)}')

all_series = []

for (store, dept), grp in pairs:
    grp = grp.sort_values('Date')
    sales = grp['Weekly_Sales'].values
    holidays = grp['IsHoliday'].values
    if len(sales) < MIN_HISTORY + VAL_WEEKS:
        continue
    all_series.append({
        'store': store, 'dept': dept,
        'sales': sales,
        'holidays': holidays,
    })

print(f'pairs with enough history: {len(all_series)}')


total pairs: 3331
pairs with enough history: 3047


## Load model


In [7]:
import torch
torch.set_float32_matmul_precision('high')

tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained('google/timesfm-2.5-200m-pytorch')
tfm.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=VAL_WEEKS,
        use_continuous_quantile_head=False,
        normalize_inputs=True,
        infer_is_positive=True,
    )
)
print('model loaded (timesfm 2.5 API)')


config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/925M [00:00<?, ?B/s]

model loaded (timesfm 2.5 API)


## Zero-shot eval


In [8]:
# feed history up to val cutoff, forecast val_weeks
histories = []
actuals = []
hol_flags = []

for s in all_series:
    sales = s['sales']
    hols = s['holidays']
    cutoff = len(sales) - VAL_WEEKS
    histories.append(sales[:cutoff].astype(np.float32))
    actuals.append(sales[cutoff:cutoff + VAL_WEEKS])
    hol_flags.append(hols[cutoff:cutoff + VAL_WEEKS])

print(f'forecasting {len(histories)} series...')


forecasting 3047 series...


In [9]:
# run in batches (memory issues)
BATCH = 256
all_forecasts = []

for start in range(0, len(histories), BATCH):
    batch_hist = histories[start:start + BATCH]
    preds, _ = tfm.forecast(horizon=VAL_WEEKS, inputs=batch_hist)
    all_forecasts.append(preds)
    if start % 1024 == 0:
        print(f'  done {start}/{len(histories)}')

forecasts = np.concatenate(all_forecasts, axis=0)
print(f'forecast shape: {forecasts.shape}')


  done 0/3047
  done 1024/3047
  done 2048/3047
forecast shape: (3047, 12)


In [10]:
# calc wmae across pairs
all_pred = forecasts.flatten()
all_true = np.concatenate(actuals)
all_hol = np.concatenate(hol_flags)

score = wmae(all_true, all_pred, all_hol)
print(f'zero-shot val WMAE: {score:.2f}')


zero-shot val WMAE: 1252.85


In [11]:
with mlflow.start_run(run_name='TimesFM_ZeroShot'):
    mlflow.log_param('model', 'timesfm-1.0-200m-pytorch')
    mlflow.log_param('horizon_len', VAL_WEEKS)
    mlflow.log_param('n_series', len(all_series))
    mlflow.log_param('freq', 2)
    mlflow.log_metric('val_wmae', score)
    mlflow.log_metric('n_predictions', len(all_pred))
    print(f'logged: wmae={score:.2f}')


logged: wmae=1252.85
🏃 View run TimesFM_ZeroShot at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/6/runs/569efb0ffb0a4f5f9daf72efe46b6459
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/6


## Try different context lengths


In [12]:
# truncate history to see if shorter/longer context matters
context_lengths = [52, 104, None]  # 1yr, 2yr, full
ctx_results = []

with mlflow.start_run(run_name='TimesFM_ContextLength'):
    for ctx in context_lengths:
        ctx_hists = []
        for h in histories:
            if ctx is not None and len(h) > ctx:
                ctx_hists.append(h[-ctx:])
            else:
                ctx_hists.append(h)

        ctx_forecasts = []
        for start in range(0, len(ctx_hists), BATCH):
            batch = ctx_hists[start:start + BATCH]
            preds, _ = tfm.forecast(horizon=VAL_WEEKS, inputs=batch)
            ctx_forecasts.append(preds)

        fc = np.concatenate(ctx_forecasts, axis=0).flatten()
        s = wmae(all_true, fc, all_hol)

        label = str(ctx) if ctx else 'full'
        mlflow.log_metric(f'wmae_ctx_{label}', s)
        ctx_results.append({'context': label, 'wmae': s})
        print(f'context={label}: wmae={s:.2f}')

ctx_df = pd.DataFrame(ctx_results)
ctx_df


context=52: wmae=1649.06
context=104: wmae=1264.08
context=full: wmae=1252.85
🏃 View run TimesFM_ContextLength at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/6/runs/9f52d9d8e7184477971568a8bb5a6898
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/6


,context,wmae
0,52,1649.059630
1,104,1264.075843
2,full,1252.846531


## Summary


In [13]:
best_ctx = ctx_df.loc[ctx_df['wmae'].idxmin()]
print(f'best context: {best_ctx["context"]}, wmae: {best_ctx["wmae"]:.2f}')
print(f'\ntimesfm is zero-shot so no training needed')
print(f'total series evaluated: {len(all_series)}')


best context: full, wmae: 1252.85

timesfm is zero-shot so no training needed
total series evaluated: 3047
